已有“raw”

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
sns.set(style="whitegrid")

import platform
system_name = platform.system()
if system_name == 'Windows':
    plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows 用黑体
elif system_name == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # Mac 用 Arial Unicode

plt.rcParams['axes.unicode_minus'] = False # 解决负号显示问题

In [2]:
file_path = r'E:\26Spring\\Final\\code\\raw.csv'

try:
    df = pd.read_csv(file_path, encoding='utf-8')
except UnicodeDecodeError:
    # 如果 utf-8 报错，尝试 gbk (Windows 创建的 csv 常用 gbk)
    df = pd.read_csv(file_path, encoding='gbk')

print(df.head())
print(df.info())
print(len(df))

   列1     列2             列3       列4     列5     列6     列7        列8  \
0  序号   股票代码           股票名称  回购数量(股)  最高回购价  最低回购价  回购平均价  回购总额(港元)   
1   1  00103           首佳科技    2.00万  3.000  2.970  2.989     5.98万   
2   2  00314           思派健康  165.60万  2.760  2.630  2.735   452.95万   
3   3  00336           华宝国际   21.00万  4.700  4.640  4.685    98.38万   
4   4  00345  VITASOY INT'L    9.40万  7.000  6.980  6.985    65.66万   

           列9  
0          日期  
1  2026-02-03  
2  2026-02-03  
3  2026-02-03  
4  2026-02-03  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64451 entries, 0 to 64450
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   列1      64451 non-null  object
 1   列2      64451 non-null  object
 2   列3      64451 non-null  object
 3   列4      64451 non-null  object
 4   列5      64451 non-null  object
 5   列6      64451 non-null  object
 6   列7      64451 non-null  object
 7   列8      64451 non-null  object
 8   列9  

In [3]:
# 1. 剔除重复的表头行
# 这里假设 '股票代码' 是列名之一，且重复行里这一列的值也是 '股票代码'
df = df[df['列2'] != '股票代码']

# 2. (重要) 将数字类型的列转换为数字
# 之前因为混入了 "回购数量(股)" 这种中文，这一列变成了文本，无法求和。
# errors='coerce' 表示如果遇到无法转换的值（比如空字符串），就变成 NaN
numeric_cols = ['回购数量(股)', '最高回购价', '最低回购价', '回购平均价', '回购总额(港元)', '日期']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.reset_index(drop=True)

print(len(df))
print(df.head())
print(df.dtypes) # 检查一下数据类型是否变回了 float/int

63187
  列1     列2             列3       列4     列5     列6     列7       列8          列9
0  1  00103           首佳科技    2.00万  3.000  2.970  2.989    5.98万  2026-02-03
1  2  00314           思派健康  165.60万  2.760  2.630  2.735  452.95万  2026-02-03
2  3  00336           华宝国际   21.00万  4.700  4.640  4.685   98.38万  2026-02-03
3  4  00345  VITASOY INT'L    9.40万  7.000  6.980  6.985   65.66万  2026-02-03
4  5  00697           首程控股  109.40万  2.010  1.990  2.008  219.65万  2026-02-03
列1    object
列2    object
列3    object
列4    object
列5    object
列6    object
列7    object
列8    object
列9    object
dtype: object


In [4]:
def clean_wan(value):
    """
    处理带有 '万' 的字符串：
    1. 去掉 '万'
    2. 转换为 float
    3. 乘以 10000
    如果无法转换（比如原本就是数字或为空），则尝试直接转换
    """
    if pd.isna(value):  # 如果是空值
        return np.nan
        
    # 确保转为字符串处理
    str_val = str(value)
    
    # 处理 '万'
    if '万' in str_val:
        # 去掉'万'，去掉可能存在的逗号，转为浮点数，再乘 10000
        number = float(str_val.replace('万', '').replace(',', ''))
        return number * 10000
    
    # 如果没有万字，尝试直接转数字
    try:
        return float(str_val)
    except ValueError:
        return np.nan # 无法转换则返回空

In [5]:
# -----------------------------------------------
# 1. 处理带“万”的列 (列4 和 列8)
# -----------------------------------------------
# 注意：请检查你的真实列名是否是 '列4'，如果之前重命名了，请换成实际中文名
df['列4'] = df['列4'].apply(clean_wan)
df['列8'] = df['列8'].apply(clean_wan)

# -----------------------------------------------
# 2. 处理纯价格列 (列5, 列6, 列7)
# -----------------------------------------------
cols_price = ['列5', '列6', '列7']
for col in cols_price:
    # errors='coerce' 意思是如果遇到无法转换的乱码，就变成 NaN (空值)，防止报错
    df[col] = pd.to_numeric(df[col], errors='coerce')

# -----------------------------------------------
# 3. 处理日期列 (列9)
# -----------------------------------------------
df['列9'] = pd.to_datetime(df['列9'])

# -----------------------------------------------
# 4. 检查结果
# -----------------------------------------------
print("转换后的数据类型：")
print(df.dtypes)
print("\n转换后的前5行数据：")
print(df.head())

转换后的数据类型：
列1            object
列2            object
列3            object
列4           float64
列5           float64
列6           float64
列7           float64
列8           float64
列9    datetime64[ns]
dtype: object

转换后的前5行数据：
  列1     列2             列3         列4    列5    列6     列7         列8         列9
0  1  00103           首佳科技    20000.0  3.00  2.97  2.989    59800.0 2026-02-03
1  2  00314           思派健康  1656000.0  2.76  2.63  2.735  4529500.0 2026-02-03
2  3  00336           华宝国际   210000.0  4.70  4.64  4.685   983800.0 2026-02-03
3  4  00345  VITASOY INT'L    94000.0  7.00  6.98  6.985   656600.0 2026-02-03
4  5  00697           首程控股  1094000.0  2.01  1.99  2.008  2196500.0 2026-02-03


In [6]:
# 建立一个映射字典
column_mapping = {
    '列1': '序号',
    '列2': '股票代码',
    '列3': '股票名称',
    '列4': '回购数量',
    '列5': '最高价',
    '列6': '最低价',
    '列7': '回购平均价', 
    '列8': '回购总额',
    '列9': '日期'
}

# 重命名
df.rename(columns=column_mapping, inplace=True)
df

,序号,股票代码,股票名称,回购数量,最高价,最低价,回购平均价,回购总额,日期
0,1,00103,首佳科技,20000.0,3.00,2.97,2.989,59800.0,2026-02-03
1,2,00314,思派健康,1656000.0,2.76,2.63,2.735,4529500.0,2026-02-03
2,3,00336,华宝国际,210000.0,4.70,4.64,4.685,983800.0,2026-02-03
3,4,00345,VITASOY INT'L,94000.0,7.00,6.98,6.985,656600.0,2026-02-03
4,5,00697,首程控股,1094000.0,2.01,1.99,2.008,2196500.0,2026-02-03
...,...,...,...,...,...,...,...,...,...
63182,63183,02348,东瑞制药,1808000.0,5.40,5.19,5.307,9595100.0,2015-01-02
63183,63184,03368,百盛集团,2000000.0,2.00,1.95,1.985,3969200.0,2015-01-02
63184,63185,03918,金界控股,600000.0,6.35,6.29,6.331,3798800.0,2015-01-02
63185,63186,08128,中国恒有源集团,1760000.0,0.37,0.36,0.361,635000.0,2015-01-02


In [7]:
HSCI_PATH = r"E:\26Spring\Final\data\HSCI.csv"

members = pd.read_csv(HSCI_PATH)
members["date"] = pd.to_datetime(members["date"], errors="coerce")
members["sid"] = members["sid"].astype(str).str.strip()

members

,date,sid
0,2005-01-03,0001.HK
1,2005-01-03,0715.HK
2,2005-01-03,0716.HK
3,2005-01-03,0728.HK
4,2005-01-03,0737.HK
...,...,...
2019619,2025-12-31,1883.HK
2019620,2025-12-31,1888.HK
2019621,2025-12-31,1896.HK
2019622,2025-12-31,1797.HK


In [8]:
# 1. 获取所有不重复的 sid
unique_sids = members['sid'].unique()

# 2. 列表推导式处理：去后缀 + 补齐5位
# 逻辑拆解：
# x.split('.')[0] -> 拿到 "." 之前的部分 (例如 "0001")
# .zfill(5)       -> 长度不足5位时在前面补0 (例如 "00001")
sid_list = [x.split('.')[0].zfill(5) for x in unique_sids]

# --- 打印检查 ---
print(f"共提取了 {len(sid_list)} 个股票代码")
print("前10个结果:", sid_list[:10])

共提取了 1126 个股票代码
前10个结果: ['00001', '00715', '00716', '00728', '00737', '00751', '00762', '00809', '00836', '00857']


In [9]:
target_col = '股票代码' 

# 1. 预处理：确保 df 里的代码也是 5 位字符串，防止 "700" 和 "00700" 匹配不上的情况
# astype(str): 强转字符串
# str.zfill(5): 不足5位补0
df[target_col] = df[target_col].astype(str).str.zfill(5)

# 2. 筛选核心代码：isin()
# 意思是：保留 df 中 [股票代码] 出现在 [sid_list] 里的那些行
filtered_df = df[df[target_col].isin(sid_list)]

# 3. 重置索引（美观）
filtered_df = filtered_df.reset_index(drop=True)

# --- 打印结果 ---
print(f"筛选前行数: {len(df)}")
print(f"筛选后行数: {len(filtered_df)}")
filtered_df

筛选前行数: 63187
筛选后行数: 42287


,序号,股票代码,股票名称,回购数量,最高价,最低价,回购平均价,回购总额,日期
0,2,00314,思派健康,1656000.0,2.76,2.63,2.735,4529500.0,2026-02-03
1,3,00336,华宝国际,210000.0,4.70,4.64,4.685,983800.0,2026-02-03
2,4,00345,VITASOY INT'L,94000.0,7.00,6.98,6.985,656600.0,2026-02-03
3,5,00697,首程控股,1094000.0,2.01,1.99,2.008,2196500.0,2026-02-03
4,6,00732,信利国际,1000000.0,1.04,1.04,1.040,1040000.0,2026-02-03
...,...,...,...,...,...,...,...,...,...
42282,63181,01555,MI能源,11430000.0,0.88,0.82,0.867,9909900.0,2015-01-02
42283,63182,02299,百宏实业,26000.0,4.00,3.99,3.994,103800.0,2015-01-02
42284,63183,02348,东瑞制药,1808000.0,5.40,5.19,5.307,9595100.0,2015-01-02
42285,63184,03368,百盛集团,2000000.0,2.00,1.95,1.985,3969200.0,2015-01-02


In [10]:
filtered_df.to_csv('em_buyback_filtered.csv', index=False, encoding='utf-8-sig')

得到em_buyback_filtered.csv